# Working with CSV and JSON: Options, Modes, and Nested Structures

**Module**: 08_data_sources_and_formats  
**Notebook**: 01_csv_and_json.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Reading and writing CSV and JSON files, handling read modes (PERMISSIVE, DROPMALFORMED, FAILFAST), and flattening nested JSON payloads.
- **Why does this topic exist?**: CSV and JSON are ubiquitous raw ingestion formats in modern data platforms.
- **What problem does it solve?**: Corrupted rows crashing pipelines or silent type mismatches.
- **Where is it used in real-world Data Engineering?**: Raw bronze-layer ingestion pipelines in data lakes.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrame basics, schemas, file paths |
| **Required depth** | Working knowledge: can read CSV/JSON safely with explicit schemas and malformed-record handling |
| **Data Engineer** | Ingest CSV/JSON with explicit schemas, validation, and predictable bad-record behavior. |
| **Senior Data Engineer** | Design ingestion contracts and diagnose silent data corruption. |
| **Spark specialist** | Reason about parser options, inference passes, and nested JSON handling. |
| **Next topic** | `08_data_sources_and_formats/02_parquet_and_orc.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

CSV and JSON are common ingestion formats but poor analytical storage. They lack types, tolerate malformed
records, and force parsing on every read. The dangerous default is schema inference: it scans the data, guesses
types, and can silently change a column from integer to string when one bad row appears.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Schema inference** | Guess types from a sample | Extra pass and unstable results |
| **Read mode** | Malformed-record policy | PERMISSIVE, DROPMALFORMED, FAILFAST |
| **Header** | First row contains names | Must be set explicitly for CSV |
| **Nested JSON** | Structs and arrays | Accessible with dot notation or `from_json` |
| **Corrupt record column** | Captures bad rows | `_corrupt_record` in permissive mode |
| **Explicit schema** | Declared types | Avoids inference and makes parsing deterministic |

### How it works, step by step

1. The reader lists files and optionally infers the schema.
2. Each file is split into partitions for parallel parsing.
3. The parser converts text fields to the declared types.
4. Malformed records are handled according to the read mode.
5. Nested JSON is materialized as structs/arrays or kept as strings.
6. The resulting DataFrame is lazy until an action or write.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Read Modes: PERMISSIVE (default) vs DROPMALFORMED vs FAILFAST

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("CSV_JSON_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Read CSV with options
sales = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/sales.csv")
sales.show(3)

# Read nested JSON from retail/order_events.json
events = spark.read.json("/opt/spark-data/retail/order_events.json")
events.printSchema()
events.select("event_id", "order.order_id", "customer.country").show(3)

---

## Experiment: Testing FAILFAST on Corrupt CSV

- **1. What are we testing?**: Attempt to read dirty CSV in FAILFAST mode.
- **2. What do we expect?**: Spark throws an exception immediately on first bad row.

In [ ]:
try:
    spark.read.option("header", True).option("mode", "FAILFAST").schema("order_id int, date date").csv("/opt/spark-data/quality/orders_dirty.csv").count()
except Exception as e:
    print(f"FAILFAST successfully threw error on malformed data: {type(e).__name__}")

**3. What happened?**: Threw exception upon parsing invalid dates/ints.  
**4. Why did it happen?**: FAILFAST stops execution immediately.  
**5. Practical Engineering Lesson**: Use PERMISSIVE with columnNameOfCorruptRecord for resilient batch ingestion.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Partner feed ingestion

- **Industry**: Retail
- **Scenario**: A retailer receives daily CSV feeds from dozens of suppliers with inconsistent headers and types.
- **Problem**: Schema inference produces different types across files, and a single malformed row can fail or silently null a column.
- **Why this concept is relevant**: An explicit schema plus a validation step makes ingestion deterministic and surfaces bad records.
- **Implementation idea**: Declare the schema, read with PERMISSIVE plus `_corrupt_record`, quarantine bad rows, and alert on counts.
- **Expected result**: A stable ingestion contract with measurable data-quality signals.
- **Alternative**: Land raw files and parse them in a managed ETL service.
- **Trade-offs**: A managed service adds governance but may not support custom validation logic.

### Use Case 2: Event JSON with evolving fields

- **Industry**: SaaS
- **Scenario**: Application events arrive as JSON lines and new optional fields are added frequently.
- **Problem**: Reading with inference makes the schema drift, and nested fields are hard to query.
- **Why this concept is relevant**: An explicit envelope schema with a raw payload column lets the team parse known fields and preserve unknown ones.
- **Implementation idea**: Read the envelope, parse the payload with `from_json`, and keep the raw string for replay.
- **Expected result**: A stable pipeline that tolerates additive schema changes.
- **Alternative**: Write events to a schema registry and use a structured format.
- **Trade-offs**: A registry improves governance but adds infrastructure and producer coordination.

---

## Business Value

- Fewer pipeline failures from malformed input.
- Faster ingestion by avoiding inference passes.
- Auditable bad-record handling.
- Clear contracts between producers and consumers.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Always provide an explicit schema in production.
- Use FAILFAST for critical feeds and quarantine for exploratory or high-volume feeds.
- Avoid `inferSchema` on large or growing datasets.
- Parse JSON once and store a columnar format for repeated reads.
- Track corrupt-record counts as a data-quality metric.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare inference with an explicit schema

**Steps**

1. Read a CSV with `inferSchema=True` and inspect the types.
2. Read the same file with an explicit schema.
3. Introduce a malformed row and compare the behavior under PERMISSIVE and FAILFAST.

**What to observe**: Inference adds a pass and can change types; the explicit schema is deterministic.

**Healthy result**: Bad rows are captured or rejected according to policy.

**Problematic result**: A malformed row silently becomes null in a critical column.

**Engineering takeaway**: Explicit schemas and corrupt-record handling are production requirements.

---

## Trade-offs and Alternatives

### Advantages

- Human-readable
- Ubiquitous
- Easy to inspect

### Limitations

- No types
- Expensive parsing
- Poor compression and predicate pushdown

### When to use it

- Landing raw data
- Partner exchanges
- Configuration files

### When not to use it

- Repeated analytical scans
- Large-scale storage
- Schema-less production contracts

### Alternatives inside Spark

- CSV/JSON readers
- from_json
- to_json

### Alternatives outside Spark

- Avro
- Parquet
- Protobuf
- Schema registries

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Read Multi-line JSON
**Problem Statement**: Set multiline option to true.  
**Input / Context**: json

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Flatten Struct
**Problem Statement**: Flatten nested customer struct.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Write JSON
**Problem Statement**: Write sales to JSON format.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The column that became a string

**Scenario**: A numeric column starts arriving with a currency symbol and the inferred type changes to string.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the inferred schema and the sample used for inference.
2. Look for malformed or formatted values in the raw data.
3. Declare the expected type and add a cleaning/validation step.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Read Multi-line JSON):
```python
print('spark.read.option("multiline", True).json(path)')
```

#### Solution 2 (Flatten Struct):
```python
events.select('customer.*').show(3)
```

#### Solution 3 (Write JSON):
```python
sales.write.mode('overwrite').json('/opt/spark-data/warehouse/sales_json')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Modes**: PERMISSIVE tracks bad rows; FAILFAST throws; DROPMALFORMED discards.
- **JSON**: Natively parsed into StructType.

### Essential Technical Terms
- `PERMISSIVE`: Default parsing mode.
- `columnNameOfCorruptRecord`: Captures raw bad string.

### Common Engineering Mistakes
- **Using text formats for petabyte analytics**: CSV/JSON lack columnar compression and predicate pushdown.

### Interview Questions on This Topic
- **Q: What are the three read modes in Spark?**
  - *A*: PERMISSIVE, DROPMALFORMED, and FAILFAST.


### More Interview Questions

- **Q: Why is schema inference risky in production?**
  - *A*: It requires an extra pass, samples the data, and can guess a different type when values change, causing silent corruption or job failures.
- **Q: What does PERMISSIVE mode do with malformed records?**
  - *A*: It sets unparseable fields to null and can capture the raw record in `_corrupt_record` when the column is included in the schema.

### What to Study Next
Open 02_parquet_and_orc.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')